# CS604 Colab launcher: pilot sweep, then full run

Runtime: **GPU** (A100 40GB, High-RAM off). Same repo / Drive / data setup as `colab_train_one-config.ipynb`.

What it does:
1. **Pilot** (`scripts/pilot_sweep.py`): short runs on a 1,000-class subset, one per value in the config's `pilot.sweep` block, scored on a labelled holdout. Writes `pilot_sweep.csv` and a `best_config.yaml` with the chosen values.
2. **Full run** (`src.train`): trains on all data with `best_config.yaml` (or `CONFIG` directly if `RUN_PILOT = False`). The epoch count is set from the pilot's measured speed so training fits `BUDGET_MIN`, with a hard stop as a safety net.
3. **Predict**: writes `val_team_{TEAM}.txt` for CodaBench.

All settings, including `img_size`, live in the YAML. The GitHub repo is private, so the clone cell asks for a Personal Access Token (hidden). Do not paste the token into the notebook text or commit it, and clear outputs before committing this notebook.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
PILOT_DATA_DIR = "/content/pilot_data"   # local disk; label files only, images are symlinked
# Must match a folder (or shortcut) in My Drive, not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
CONFIG = "configs/lora_r16_336px_qkv-proj-fc_lr-sweep.yaml"     # must have a pilot: block if RUN_PILOT is True
RUN_PILOT = True
TEAM = "08"
BUDGET_MIN = 144     # training budget: 2 hr x 48GB, taken as 144 min on an A100 40GB
SAFETY = 0.95        # plan epochs to use at most 95% of the budget

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin -q
    !git -C {REPO_DIR} checkout -q {BRANCH}
    if token:
        !git -C {REPO_DIR} pull -q {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {clone_url} {REPO_DIR}

# Keep the token out of .git/config so it can't leak into later outputs.
!git -C {REPO_DIR} remote set-url origin {REPO_URL}
del token, clone_url

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -q -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
print("Writing runs to", out.resolve())

## 4. Download data to `/content/data`

Skipped if the data is already unpacked in this session. If the download fails with 401, run the optional login cell first.

In [ ]:
# Optional: only if the dataset download asks for a Hugging Face login
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
if (Path(DATA_DIR) / "train.txt").exists():
    print("Data already in", DATA_DIR)
else:
    !python scripts/download_data.py --dest {DATA_DIR}

## Review settings (table)

`CONFIG` merged with `shared_vit-b16_augreg2_defaults.yaml`. The `pilot` row shows what the pilot will sweep and its pilot-only overrides; the full run uses the other rows.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

chosen = load_config(CONFIG)
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT)
display(pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"]))
if RUN_PILOT and not chosen.get("pilot"):
    raise ValueError(f"{CONFIG} has no pilot: block. Add one or set RUN_PILOT = False.")

## 5. Pilot sweep

About 5 to 7 min per value at 336 on an A100 with the default 1,000-class subset and 3 epochs. The pilot is for **choosing** parameters only; its numbers are not comparable to full-data holdout scores.

In [ ]:
import json

FULL_CONFIG = CONFIG
if RUN_PILOT:
    !python scripts/pilot_sweep.py --config {CONFIG} --data_root {DATA_DIR} \
        --output_dir {DRIVE_OUT} --pilot_data_dir {PILOT_DATA_DIR}
    sweep_dir = sorted(Path(DRIVE_OUT).glob("pilot_*"), key=lambda p: p.stat().st_mtime)[-1]
    print("Pilot folder:", sweep_dir)
    display(pd.read_csv(sweep_dir / "pilot_sweep.csv"))
    summary = json.loads((sweep_dir / "pilot_summary.json").read_text())
    print("Chosen:", summary["chosen"], "|", summary["reason"])
    FULL_CONFIG = str(sweep_dir / "best_config.yaml")
else:
    print("Skipping pilot; full run uses", CONFIG)

### Pilot curves

Look at the shape, not just the final number. An accuracy dip partway through means the lr is too high even if it recovers. If the chosen value sits at the edge of the sweep (e.g. the highest lr), extend the sweep before trusting it.

In [ ]:
import matplotlib.pyplot as plt

if RUN_PILOT:
    fig, ax = plt.subplots(figsize=(6, 4))
    for run in sorted(p for p in sweep_dir.iterdir() if (p / "epoch_log.csv").exists()):
        log = pd.read_csv(run / "epoch_log.csv")
        cfg_run = load_config(run / "config.yaml")
        label = ", ".join(f"{k}={cfg_run.get(k)}" for k in summary["chosen"])
        ax.plot(log["epoch"], log["val_acc"], marker="o", label=label)
    ax.set_xlabel("epoch"); ax.set_ylabel("pilot holdout top-1"); ax.legend(); ax.grid(alpha=0.3)
    plt.show()

## 6. Full run (with time budget)

**Planning:** the pilot logs training seconds per epoch on a known number of images, which gives images/sec on this GPU. From that, the cell below estimates minutes per full epoch and sets `epochs` so training fits `SAFETY × BUDGET_MIN`. Planning the epoch count (rather than cutting a run short) keeps the cosine schedule complete. Without a pilot, it uses the config's `epochs` and only warns.

**Safety net:** the run is wrapped in `timeout`, which interrupts it at `BUDGET_MIN` plus a few minutes for model loading. `last.pt` is saved after every epoch, so an interrupted run keeps its last completed epoch, but `run_stats.txt` won't be written and the schedule won't have finished, so treat that as a failed run and reduce `epochs`.

The course val split has no labels, so there is no per-epoch accuracy and no `best.pt`; predict from `last.pt`.

In [ ]:
import math

full_cfg = load_config(FULL_CONFIG)
n_train = sum(1 for _ in open(Path(DATA_DIR) / "train.txt"))
epochs = int(full_cfg.epochs)

if RUN_PILOT:
    fit_images = summary["pilot_data"]["fit_images"]
    epoch_times = []
    for r in summary["results"]:
        log = pd.read_csv(Path(r["run_dir"]) / "epoch_log.csv")
        # skip epoch 1 when possible: it includes first-pass disk reads and warm-up
        times = log["epoch_time_s"].iloc[1:] if len(log) > 1 else log["epoch_time_s"]
        epoch_times.extend(times.tolist())
    img_per_s = fit_images / float(pd.Series(epoch_times).median())
    min_per_epoch = n_train / img_per_s / 60
    fit_epochs = max(1, math.floor(SAFETY * BUDGET_MIN / min_per_epoch))
    print(f"Pilot speed: {img_per_s:.0f} img/s -> {min_per_epoch:.1f} min per full epoch ({n_train:,} images)")
    print(f"Config epochs: {epochs}; fits in {SAFETY:.0%} of {BUDGET_MIN} min: {fit_epochs}")
    epochs = min(epochs, fit_epochs)
else:
    print(f"No pilot timing; using config epochs={epochs}. Check epoch_log.csv after epoch 1.")

planned_min = epochs * min_per_epoch if RUN_PILOT else float("nan")
print(f"Planned: {epochs} epochs, about {planned_min:.0f} min of training (budget {BUDGET_MIN} min)")

In [ ]:
HARD_STOP_MIN = BUDGET_MIN + 5   # small allowance for model loading before training starts
print("Full run config:", FULL_CONFIG, "| epochs:", epochs, "| hard stop:", HARD_STOP_MIN, "min")
!timeout --signal=INT {HARD_STOP_MIN}m python -m src.train --config {FULL_CONFIG} \
    --data_root {DATA_DIR} --output_dir {DRIVE_OUT} --epochs {epochs}

In [ ]:
run_dir = sorted(Path(DRIVE_OUT).glob(f"{full_cfg.method}_*"), key=lambda p: p.stat().st_mtime)[-1]
print("Run folder:", run_dir)
log = pd.read_csv(run_dir / "epoch_log.csv")
display(log)

train_min = log["epoch_time_s"].sum() / 60
print(f"Training time: {train_min:.1f} min of {BUDGET_MIN} min budget ({train_min / BUDGET_MIN:.0%})")
if train_min > BUDGET_MIN:
    print("WARNING: over budget. Reduce epochs and rerun.")
if len(log) < epochs or not (run_dir / "run_stats.txt").exists():
    print(f"WARNING: run stopped early ({len(log)}/{epochs} epochs); likely hit the hard stop.")
else:
    print((run_dir / "run_stats.txt").read_text())

## 7. Predict for CodaBench

In [ ]:
ckpt = run_dir / ("best.pt" if (run_dir / "best.pt").exists() else "last.pt")
!python -m src.predict --config {run_dir / "config.yaml"} --ckpt {ckpt} --split val --team {TEAM}
print("Then add a row to report/experiment_index.csv and a note in report/progress_log.md")

## 8. (Optional) Shut down the runtime

In [ ]:
from google.colab import runtime

runtime.unassign()